# Clustering Preserving QAT

## Install TensorFlow Model Optimization Toolkit
* 설치 완료 후 반드시 Runtime 재시작!
    * '런타임' > '세션 다시 시작' 메뉴 선택'

In [1]:
!pip install tensorflow-model-optimization

## Mount Google driver

In [2]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('g-drive mounted.')
    colab=True
except:
    print('local drive.')
    colab =False

Mounted at /content/drive
g-drive mounted.


## Import Module

In [3]:
import tensorflow as tf
import numpy as np

import tensorflow_model_optimization as tfmot
from tensorflow_model_optimization.python.core.keras.compat import keras

import tempfile

print(tf.__version__)
print(np.__version__)

2.19.0
1.26.4


## Load Dataset

In [4]:
(train_images, train_labels), (test_images, test_labels) = keras.datasets.mnist.load_data()

train_images = (train_images / 255.0).astype(np.float32)
test_images = (test_images / 255.0).astype(np.float32)

11490434/11490434 [==============================] - 0s 0us/step


## Load Baseline Model for MNIST (CNN)

In [5]:
model = tf.keras.models.load_model('/content/drive/MyDrive/files/save/baseline_model.h5')
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 reshape (Reshape)           (None, 28, 28, 1)         0         
                                                                 
 conv2d (Conv2D)             (None, 26, 26, 32)        320       
                                                                 
 max_pooling2d (MaxPooling2  (None, 13, 13, 32)        0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 11, 11, 16)        4624      
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 5, 5, 16)          0         
 g2D)                                                            
                                                                 
 flatten (Flatten)           (None, 400)               0

In [6]:
_, baseline_model_accuracy = model.evaluate(test_images, test_labels, verbose=0)

print('Baseline test accuracy:', baseline_model_accuracy)

Baseline test accuracy: 0.9900000095367432


## Cluster and fine-tune
* number of cluster : 8
* cluster centroids init : K-means++

In [7]:
clustering_params = {
  'number_of_clusters': 8,
  'cluster_centroids_init': tfmot.clustering.keras.CentroidInitialization.KMEANS_PLUS_PLUS,
  'cluster_per_channel' : True
}

clustered_model = tfmot.clustering.keras.cluster_weights(model, **clustering_params)

# Use smaller learning rate for fine-tuning
clustered_model.compile(
  loss=keras.losses.SparseCategoricalCrossentropy(),
  optimizer=keras.optimizers.Adam(learning_rate=1e-5),
  metrics=['accuracy'])

clustered_model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 cluster_reshape (ClusterWe  (None, 28, 28, 1)         0         
 ights)                                                          
                                                                 
 cluster_conv2d (ClusterWei  (None, 26, 26, 32)        864       
 ghts)                                                           
                                                                 
 cluster_max_pooling2d (Clu  (None, 13, 13, 32)        0         
 sterWeights)                                                    
                                                                 
 cluster_conv2d_1 (ClusterW  (None, 11, 11, 16)        9360      
 eights)                                                         
                                                                 
 cluster_max_pooling2d_1 (C  (None, 5, 5, 16)          0

## Fine tune the model for clustering

In [8]:
clustered_model.fit(
  train_images,
  train_labels,
  epochs=3,
  validation_split=0.1)

Epoch 1/3
1688/1688 [==============================] - 61s 33ms/step - loss: 0.0093 - accuracy: 0.9972 - val_loss: 0.0341 - val_accuracy: 0.9920
Epoch 2/3
1688/1688 [==============================] - 56s 33ms/step - loss: 0.0052 - accuracy: 0.9986 - val_loss: 0.0325 - val_accuracy: 0.9933
Epoch 3/3
1688/1688 [==============================] - 54s 32ms/step - loss: 0.0037 - accuracy: 0.9991 - val_loss: 0.0335 - val_accuracy: 0.9930


## Cluster 확인

In [9]:
def print_model_weight_clusters(model):
    for layer in model.layers:
        if isinstance(layer, keras.layers.Wrapper):
            weights = layer.trainable_weights
        else:
            weights = layer.weights
        for weight in weights:
            # ignore auxiliary quantization weights
            if "quantize_layer" in weight.name:
                continue
            if "kernel" in weight.name:
                unique_count = len(np.unique(weight))
                print(
                    f"{layer.name}/{weight.name}: {unique_count} clusters "
                )

In [10]:
stripped_clustered_model = tfmot.clustering.keras.strip_clustering(clustered_model)

print_model_weight_clusters(stripped_clustered_model)

conv2d/kernel:0: 256 clusters 
conv2d_1/kernel:0: 128 clusters 
dense/kernel:0: 8 clusters 
dense_1/kernel:0: 8 clusters 


In [11]:
_, clustered_model_accuracy = clustered_model.evaluate(
  test_images, test_labels, verbose=0)

print('Baseline test accuracy:', baseline_model_accuracy)
print('Clustered test accuracy:', clustered_model_accuracy)

Baseline test accuracy: 0.9900000095367432
Clustered test accuracy: 0.9922000169754028


## QAT VS CQAT
* QAT : training 과정에서 cluster 파괴
* CQAT : training 과정에서도 cluster 유지

In [12]:
# QAT
qat_model = tfmot.quantization.keras.quantize_model(stripped_clustered_model)

qat_model.compile(optimizer='adam',
              loss=keras.losses.SparseCategoricalCrossentropy(),
              metrics=['accuracy'])

qat_model.summary()

print('Train QAT model:')
qat_model.fit(train_images, train_labels, batch_size=128, epochs=1, validation_split=0.1)

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 quantize_layer (QuantizeLa  (None, 28, 28)            3         
 yer)                                                            
                                                                 
 quant_reshape (QuantizeWra  (None, 28, 28, 1)         1         
 pperV2)                                                         
                                                                 
 quant_conv2d (QuantizeWrap  (None, 26, 26, 32)        387       
 perV2)                                                          
                                                                 
 quant_max_pooling2d (Quant  (None, 13, 13, 32)        1         
 izeWrapperV2)                                                   
                                                                 
 quant_conv2d_1 (QuantizeWr  (None, 11, 11, 16)        4

In [13]:
# CQAT
quant_aware_annotate_model = tfmot.quantization.keras.quantize_annotate_model(
              stripped_clustered_model)
cqat_model = tfmot.quantization.keras.quantize_apply(
              quant_aware_annotate_model,
              tfmot.experimental.combine.Default8BitClusterPreserveQuantizeScheme())

cqat_model.compile(optimizer='adam',
              loss=keras.losses.SparseCategoricalCrossentropy(),
              metrics=['accuracy'])

cqat_model.summary()

print('Train CQAT Model:')
cqat_model.fit(train_images, train_labels, batch_size=128, epochs=1, validation_split=0.1)

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 quantize_layer_1 (Quantize  (None, 28, 28)            3         
 Layer)                                                          
                                                                 
 quant_reshape (QuantizeWra  (None, 28, 28, 1)         1         
 pperV2)                                                         
                                                                 
 quant_conv2d (QuantizeWrap  (None, 26, 26, 32)        1219      
 perV2)                                                          
                                                                 
 quant_max_pooling2d (Quant  (None, 13, 13, 32)        1         
 izeWrapperV2)                                                   
                                                                 
 quant_conv2d_1 (QuantizeWr  (None, 11, 11, 16)        1

422/422 [==============================] - 46s 94ms/step - loss: 0.0071 - accuracy: 0.9983 - val_loss: 0.0312 - val_accuracy: 0.9925


## Cluster수 확인

In [14]:
print("CQAT Model clusters:")
print_model_weight_clusters(cqat_model)
print()
print("QAT Model clusters:")
print_model_weight_clusters(qat_model)

CQAT Model clusters:
quant_conv2d/conv2d/kernel:0: 254 clusters 
quant_conv2d_1/conv2d_1/kernel:0: 128 clusters 
quant_dense/dense/kernel:0: 8 clusters 
quant_dense_1/dense_1/kernel:0: 8 clusters 

QAT Model clusters:
quant_conv2d/conv2d/kernel:0: 288 clusters 
quant_conv2d_1/conv2d_1/kernel:0: 4608 clusters 
quant_dense/dense/kernel:0: 46702 clusters 
quant_dense_1/dense_1/kernel:0: 1187 clusters 


## CQAT 모델의 장점 : 효율적인 압축 가능

In [15]:
import zipfile
import os

def get_gzipped_model_size(file):
  # It returns the size of the gzipped model in kilobytes.

  _, zipped_file = tempfile.mkstemp('.zip')
  with zipfile.ZipFile(zipped_file, 'w', compression=zipfile.ZIP_DEFLATED) as f:
    f.write(file)

  return os.path.getsize(zipped_file)/1000

In [20]:
litert_model_path = "/content/drive/MyDrive/files/save/"

In [25]:
import pathlib
models_dir = pathlib.Path(litert_model_path)
models_dir.mkdir(exist_ok=True)

# QAT model
converter = tf.lite.TFLiteConverter.from_keras_model(qat_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

qat_tflite_model = converter.convert()
qat_model_file = litert_model_path + 'mnist_cqat_x_model.tflite'

with open(qat_model_file, 'wb') as f:
    f.write(qat_tflite_model)

# CQAT model
converter = tf.lite.TFLiteConverter.from_keras_model(cqat_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

cqat_tflite_model = converter.convert()
cqat_model_file = litert_model_path + 'mnist_cqat_model.tflite'

with open(cqat_model_file, 'wb') as f:
    f.write(cqat_tflite_model)

print("QAT model size: ", get_gzipped_model_size(qat_model_file), ' KB')
print("CQAT model size: ", get_gzipped_model_size(cqat_model_file), ' KB')

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:854: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:854: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


QAT model size:  50.849  KB
CQAT model size:  29.687  KB


In [26]:
def eval_model(interpreter):
  input_details = interpreter.get_input_details()[0]
  output_details = interpreter.get_output_details()[0]
  input_index = input_details["index"]
  output_index = output_details["index"]

  prediction_digits = []
  for i, test_image in enumerate(test_images):
    if input_details['dtype'] == np.int8:
      input_scale, input_zero_point = input_details["quantization"]
      test_image = test_image / input_scale + input_zero_point

    test_image = np.expand_dims(test_image, axis=0).astype(input_details['dtype'])
    interpreter.set_tensor(input_index, test_image)

    interpreter.invoke()

    output = interpreter.get_tensor(output_index)
    digit = np.argmax(output)
    prediction_digits.append(digit)

  prediction_digits = np.array(prediction_digits)
  accuracy = (prediction_digits == test_labels).mean()
  return accuracy

In [27]:
interpreter = tf.lite.Interpreter(cqat_model_file)
interpreter.allocate_tensors()

cqat_test_accuracy = eval_model(interpreter)

print('Clustered and quantized TFLite test_accuracy :', cqat_test_accuracy)
print('Baseline model test_accuracy :', baseline_model_accuracy)

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Clustered and quantized TFLite test_accuracy : 0.9922
Baseline model test_accuracy : 0.9900000095367432
